# Neutral Flux Limiter Diagnostics Verification

Scratch notebook for inspecting `/neutral_flux_limiter_diagnostics` from a diagnostics-only run. The saved diagnostics are used as comparison targets; recomputation uses solution fields, gradients, parameters, atomic data, and neutral diffusion with neutral-neutral collisions.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from hdg_postprocess.api.setup import make_atomic_parameters, make_neutral_diffusion_parameters
from hdg_postprocess.core.solution.neutral_flux_limiter import compare_diagnostics_only_runs, split_solution_file_path
from hdg_postprocess.formats.load_from_file import load_HDG_solution_from_file

In [ ]:
repo = Path.cwd()
diag_file = repo / "demos/data/solutions/limiter_case/steady/Sol2D_WEST_60527_P8_DPe0.125E+01_DPai0.314E+06_DPae0.105E+08.h5"

# Set this to a matching neutral_flux_limiter='off' run when available.
off_file = None

atomic_dir = repo / "demos/data/atomic"
atol = 1.0e-10
rtol = 1.0e-8

In [ ]:
diag_path, diag_base = split_solution_file_path(diag_file)
solution = load_HDG_solution_from_file(diag_path, diag_base)

atomic = make_atomic_parameters(data_dir=atomic_dir)
neutral_diffusion = make_neutral_diffusion_parameters()
solution.additional_parameters.set_atomic(atomic)
solution.additional_parameters.set_neutral_diffusion(neutral_diffusion, solution.parameters["adimensionalization"])

summary = solution.neutrals.limiter_diagnostic_summary()
for key, item in summary.items():
    print(key, item)

In [ ]:
if off_file is not None:
    off_report = compare_diagnostics_only_runs(off_file, diag_file, atol=1.0e-12, rtol=1.0e-10)
    for name, metrics in off_report.items():
        print(name, metrics)
else:
    print("No off_file configured; skipping off-vs-diagnostics-only solution comparison.")

In [ ]:
report = solution.neutrals.verify_limiter_diagnostics(
    atomic_parameters=atomic,
    neutral_diffusion_parameters=neutral_diffusion,
    atol=atol,
    rtol=rtol,
    strict=False,
)
for field, metrics in report.items():
    print("\n", field)
    if isinstance(metrics, dict) and "passed" in metrics:
        print(metrics)
    else:
        for name, item in metrics.items():
            print(name, item)

In [ ]:
fields = ["phi", "activation_ratio", "Gamma_unlim", "Gamma_max", "Dnn", "D_eff"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
for ax, name in zip(axes.ravel(), fields):
    values = solution.neutrals.limiter_diagnostic(name, view="node")
    solution.mesh.plot.full(ax=ax, data=values, connectivity=solution.mesh.global_state.connectivity, label=name)
plt.show()